In [1]:
#Step1
import pandas as pd
import pandasql as ps
import pixiedust
import sys
sys.path.append('..')
import util
import etl
import pyarrow.parquet as pq
import pyarrow as pa
!pip install duckdb
import duckdb
import inspect

# Get the path of the imported module (etl.py)
etl_module_path = inspect.getfile(etl)
print("Path of the imported ETL module (etl.py):", etl_module_path)
util.usedatabase(spark, "real_world_data_jun_2022")

pixiedust.enableJobMonitor()

con = duckdb.connect()

Pixiedust database opened successfully
Table VERSION_TRACKER created successfully
Table METRICS_TRACKER created successfully

Share anonymous install statistics? (opt-out instructions)

PixieDust will record metadata on its environment the next time the package is installed or updated. The data is anonymized and aggregated to help plan for future releases, and records only the following values:

{
   "data_sent": currentDate,
   "runtime": "python",
   "application_version": currentPixiedustVersion,
   "space_id": nonIdentifyingUniqueId,
   "config": {
       "repository_id": "https://github.com/ibm-watson-data-lab/pixiedust",
       "target_runtimes": ["Data Science Experience"],
       "event_id": "web",
       "event_organizer": "dev-journeys"
   }
}
You can opt out by calling pixiedust.optOut() in a new cell.


Pixiedust runtime updated. Please restart kernel
Table SPARK_PACKAGES created successfully
Table USER_PREFERENCES created successfully
Table service_connections created successfully
Please see https://github.com/pypa/pip/issues/5599 for advice on fixing the underlying issue.
To avoid this problem you can invoke Python with '-m pip' instead of running pip directly.
     |████████████████████████████████| 18.1 MB 5.8 MB/s eta 0:00:01
Path of the imported ETL module (etl.py): /home/o_suchsi/work/Oklahoma State/Priya/epilepsy/etl.py
Using real_world_data_jun_2022 ....
Successfully enabled Spark Job Progress Monitor


Exception in thread Thread-6:
Traceback (most recent call last):
  File "/opt/conda/lib/python3.7/threading.py", line 926, in _bootstrap_inner
    self.run()
  File "/opt/conda/lib/python3.7/threading.py", line 870, in run
    self._target(*self._args, **self._kwargs)
  File "/opt/conda/lib/python3.7/site-packages/pixiedust/utils/sparkJobProgressMonitor.py", line 47, in startSparkJobProgressMonitor
    progressMonitor = SparkJobProgressMonitor()
  File "/opt/conda/lib/python3.7/site-packages/pixiedust/utils/sparkJobProgressMonitor.py", line 174, in __init__
    self.addSparkListener()
  File "/opt/conda/lib/python3.7/site-packages/pixiedust/utils/sparkJobProgressMonitor.py", line 203, in addSparkListener
    _env.getTemplate("sparkJobProgressMonitor/addSparkListener.scala").render()
  File "/opt/conda/lib/python3.7/site-packages/IPython/core/interactiveshell.py", line 2352, in run_cell_magic
    result = fn(*args, **kwargs)
  File "</opt/conda/lib/python3.7/site-packages/decorator.py:d

In [4]:
#Testing since extra
Como_Result_Final_old = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/control_comorbidity_Latest")

▸,:,


AnalysisException: 'Path does not exist: file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/control_comorbidity_Latest;'

In [2]:
#Reading-> Commo-Control-> deleted
Como_Result_Final1 = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/control_comorbidity_Paired")

In [3]:
Como_Result_Final1_cnt = Como_Result_Final1.count()
print("count control", Como_Result_Final1_cnt)

count control 20499629


In [4]:
#Remove Duplicates
Como_Result_Final1 = Como_Result_Final1.distinct()

In [5]:
Como_Result_Final1_cnt = Como_Result_Final1.count()
print("count control", Como_Result_Final1_cnt)

count control 20499629


In [6]:
Como_Result_Final1.printSchema()

▸,:,


root
 |-- personid: string (nullable = true)
 |-- comorbidityid: string (nullable = true)



In [59]:
Como_Result_Final1.createOrReplaceTempView('Epilepsy_Control_Como')

In [ ]:
Como_RecordCount_Control = spark.sql(""" 
    SELECT
        count(personid) as COUNT         
    FROM
        Epilepsy_Control_Como
""")
Como_RecordCount_Control.show()

In [ ]:
Como_Result_Final1.show(100, truncate=False)

In [133]:
#Reading-> Commo-Cohort
Como_Result_Final2 = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/cohort_comorbidity_Paired")

In [134]:
Como_Result_Final2_cnt = Como_Result_Final2.count()
print("Cohort_Paired_cnt", Como_Result_Final2_cnt)

Cohort_Paired_cnt 7930014


In [135]:
Como_Result_Final2 = Como_Result_Final2.distinct()

In [136]:
Como_Result_Final2_cnt = Como_Result_Final2.count()
print("Cohort_Paired_cnt", Como_Result_Final2_cnt)

Cohort_Paired_cnt 7930014


In [19]:
#Reading-> Commo-Cohort after splitting commas
Cohort_Paired = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Cohort_Paired_Commo_RemComma")

In [20]:
Cohort_Paired_cnt = Cohort_Paired.count()
print("Cohort_Paired_cnt", Cohort_Paired_cnt)

Cohort_Paired_cnt 7931635


In [22]:
Cohort_Paired = Cohort_Paired.distinct()

In [23]:
Cohort_Paired_cnt = Cohort_Paired.count()
print("Cohort_Paired_cnt", Cohort_Paired_cnt)

Cohort_Paired_cnt 7929539


In [61]:
Como_Result_Final2.createOrReplaceTempView('Epilepsy_Cohort_Como')

In [62]:
##############################################Final dictionary to be used for control commo conversion process #########################
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

# Read the CSV file into a DataFrame
df = spark.read.csv("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/ICDconvertion.csv", header=True, inferSchema=True)

# Define a UDF to cast the "icd9cm" column to StringType
def cast_to_string(icd9cm):
    return str(icd9cm)

# Register the UDF
cast_to_string_udf = F.udf(cast_to_string, StringType())

# Apply the UDF to cast "icd9cm" to StringType
df = df.withColumn("icd9cm", cast_to_string_udf(df["icd9cm"]))

# Convert DataFrame to a Pandas DataFrame
pandas_df = df.toPandas()

# Convert Pandas DataFrame to a dictionary
icd_dict = pandas_df.set_index("icd9cm")["icd10cm"].to_dict()

# Show the resulting dictionary
print(icd_dict)

{'10': 'A000', '11': 'A001', '19': 'A009', '20': 'A0100', '21': 'A011', '22': 'A012', '23': 'A013', '29': 'A014', '30': 'A020', '31': 'A021', '320': 'A360', '321': 'A361', '322': 'A3689', '323': 'A362', '324': 'A0224', '329': 'A369', '38': 'A028', '39': 'A029', '40': 'A030', '41': 'A031', '42': 'B20', '43': 'A033', '48': 'A880', '49': 'A039', '50': 'A050', '51': 'A051', '52': 'A052', '53': 'A058', '54': 'A053', '581': 'A055', '589': 'A058', '59': 'A059', '60': 'A060', '61': 'A90', '62': 'A062', '63': 'A064', '64': 'A852', '65': 'A066', '66': 'A067', '68': 'A0689', '69': 'A069', '70': 'A070', '71': 'A829', '72': 'A073', '73': 'A078', '74': 'A072', '75': 'B2790', '78': 'A078', '79': 'A079', '800': 'A044', '801': 'A040', '802': 'A041', '803': 'A042', '804': 'A043', '809': 'A044', '81': 'A048', '82': 'A048', '83': 'A048', '841': 'B519', '842': 'B529', '843': 'B530', '844': 'B538', '845': 'B529', '846': 'B54', '847': 'B538', '849': 'B528', '85': 'A049', '861': 'B575', '862': 'B571', '863': 

In [63]:
Como_Result_Test = spark.sql(""" 
    SELECT
        DISTINCT
        comorbidityid   
    FROM
        Epilepsy_Control_Como
""")
Como_Result_Test.show()
# Collect the distinct comorbidityid values into a list
comorbidityid_list = Como_Result_Test.rdd.flatMap(lambda x: x).collect()

# Show the distinct comorbidityid values
print(comorbidityid_list)

+-------------+
|comorbidityid|
+-------------+
|      Z87.820|
|        850.9|
|       E83.42|
|       Z76.89|
|        959.4|
|      F12.929|
|       810.03|
|     S83.231A|
|       787.20|
|    133933007|
|       H52.02|
|       H55.81|
|       M62.89|
|        286.9|
|          Z21|
|        840.8|
|       K85.90|
|       M51.87|
|       M23.92|
|        G03.9|
+-------------+
only showing top 20 rows

['Z87.820', '850.9', 'E83.42', 'Z76.89', '959.4', 'F12.929', '810.03', 'S83.231A', '787.20', '133933007', 'H52.02', 'H55.81', 'M62.89', '286.9', 'Z21', '840.8', 'K85.90', 'M51.87', 'M23.92', 'G03.9', 'M06.09', '803.00', 'N99.89', 'E937.8', 'M24.011', 'M77.40', 'S62.350A', '415.11', 'E006.9', 'M43.10', 'V14.2', 'N81.4', 'B17.9', '732.1', '521.81', '807.05', '783.43', 'T46.0X4A', 'Z51.11', 'Z45.89', 'N35.911', '110.9', 'S60.021A', '536.8', 'Q27.30', '958.3', 'V42.3', 'S60.012S', 'Z59.6', '733.01', 'F15.11', 'S14.125A', 'M50.31', 'V17.1', 'N60.31', 'S63.681A', 'M67.441', 'J30.9, R09.82'

In [64]:
Como_Result_Test = spark.sql(""" 
    SELECT
        DISTINCT
        comorbidityid   
    FROM
        Epilepsy_Cohort_Como
""")
Como_Result_Test.show()
# Collect the distinct comorbidityid values into a list
comorbidityid_list1 = Como_Result_Test.rdd.flatMap(lambda x: x).collect()

# Show the distinct comorbidityid values
print(comorbidityid_list1)

+-------------+
|comorbidityid|
+-------------+
|       M23.92|
|      Z87.820|
|     S83.231A|
|       E83.42|
|        850.9|
|       787.20|
|       Z76.89|
|        R51.0|
|        V14.2|
|       H55.81|
|       M62.89|
|       Z51.11|
|       M06.09|
|        V17.1|
|        Z59.6|
|       K85.90|
|          Z21|
|       M50.31|
|        N81.4|
|       997.62|
+-------------+
only showing top 20 rows

['M23.92', 'Z87.820', 'S83.231A', 'E83.42', '850.9', '787.20', 'Z76.89', 'R51.0', 'V14.2', 'H55.81', 'M62.89', 'Z51.11', 'M06.09', 'V17.1', 'Z59.6', 'K85.90', 'Z21', 'M50.31', 'N81.4', '997.62', 'S60.021A', '959.4', 'M67.441', '521.81', 'M43.10', 'K66.8', '286.9', 'N35.911', 'S59.919A', 'M86.8X0', 'S02.610A', 'F12.929', '803.00', '840.8', 'M31.6', 'N99.89', 'H35.32', 'S63.592A', 'C84.60', '783.43', '536.8', 'O36.8990', 'B17.9', '66857006', 'Q27.30', 'E006.9', 'Q79.9', 'M77.40', 'S52.209D', '191.9', 'L89.302', 'Y90.4', 'V57.6XXA', '458.29', 'Z45.89', 'F15.11', 'G03.9', 'S63.681A', 'J2

In [67]:
# Combine the lists and extract distinct values
combined_and_distinct = list(set(comorbidityid_list + comorbidityid_list1))

# Initialize a counter for items containing commas
comma_count = 0

# Iterate through the list and print values containing commas
for item in combined_and_distinct:
    if ',' in item:
        print(item)
        comma_count += 1

# Print the total number of items in the combined_and_distinct list
print("Total number of distinct items:", len(combined_and_distinct))
print("Number of items containing commas:", comma_count)

H52.03, H52.203
E11.22, I12.9, N18.30
E08.621, L97.529, L97.519
M25.60, Z74.09
K90.9, R19.7
S63.502A, S66.912A
S30.861A, W57.XXXA
H02.883, H02.886
S42.001A, S42.002A
S02.40FA, S02.32XA, S02.82XA, S02.40DA
M79.601, R20.0
M54.2, G89.28, Z98.890
M25.851, M25.852
E66.09, Z68.32
O34.11, D25.9
E66.1, Z68.33
E10.22, N18.6, Z99.2
Z98.41, Z96.1
M25.641, M25.642
E10.29, R80.9
D25.1, D25.0, D25.2
S27.808A, S20.212A
R11.2, F12.90
T84.028A, Z96.659
P78.89, K90.49
I63.9, R29.810
C50.919, Z15.02, Z15.89, Z15.09
E11.69, B35.1
S02.40CA, S02.40DA
S32.10XA, S32.2XXA
O26.899, E86.0
S62.101G, S62.102G
R45.4, T50.Z95A
M25.551, G89.29
S60.945A, L08.9
M79.671, M79.672
E11.40, Z79.4
R10.30, G89.29
S61.459A, W54.0XXA
H52.202, H52.12
S22.41XA, S27.0XXA
S49.80XA, T50.Z95A
N18.3, D63.1
S02.81XB, S02.82XB
S06.9X9D, G31.84
Z74.09, Z74.1
K59.03, T40.2X5A
M25.529, G89.29
H54.40, H57.12
T23.272D, T23.202D
S01.81XD, S01.112D
O99.519, J45.909
F33.3, F06.1
E66.01, Z68.35
M79.645, G89.29
S02.92XB, V89.2XXA
I63.9, R48.2
T23

In [65]:
# Combine the lists and extract distinct values
combined_and_distinct = list(set(comorbidityid_list + comorbidityid_list1))
# Iterate through the list and print values containing commas
for item in combined_and_distinct:
    if ',' in item:
        print(item)

H52.03, H52.203
E11.22, I12.9, N18.30
E08.621, L97.529, L97.519
M25.60, Z74.09
K90.9, R19.7
S63.502A, S66.912A
S30.861A, W57.XXXA
H02.883, H02.886
S42.001A, S42.002A
S02.40FA, S02.32XA, S02.82XA, S02.40DA
M79.601, R20.0
M54.2, G89.28, Z98.890
M25.851, M25.852
E66.09, Z68.32
O34.11, D25.9
E66.1, Z68.33
E10.22, N18.6, Z99.2
Z98.41, Z96.1
M25.641, M25.642
E10.29, R80.9
D25.1, D25.0, D25.2
S27.808A, S20.212A
R11.2, F12.90
T84.028A, Z96.659
P78.89, K90.49
I63.9, R29.810
C50.919, Z15.02, Z15.89, Z15.09
E11.69, B35.1
S02.40CA, S02.40DA
S32.10XA, S32.2XXA
O26.899, E86.0
S62.101G, S62.102G
R45.4, T50.Z95A
M25.551, G89.29
S60.945A, L08.9
M79.671, M79.672
E11.40, Z79.4
R10.30, G89.29
S61.459A, W54.0XXA
H52.202, H52.12
S22.41XA, S27.0XXA
S49.80XA, T50.Z95A
N18.3, D63.1
S02.81XB, S02.82XB
S06.9X9D, G31.84
Z74.09, Z74.1
K59.03, T40.2X5A
M25.529, G89.29
H54.40, H57.12
T23.272D, T23.202D
S01.81XD, S01.112D
O99.519, J45.909
F33.3, F06.1
E66.01, Z68.35
M79.645, G89.29
S02.92XB, V89.2XXA
I63.9, R48.2
T23

In [66]:
##################################Final Package to be used to normalize commorbidity for control group #########################
from pyspark.sql.functions import col, udf, split, regexp_replace, explode
from pyspark.sql.types import StringType
from pyspark.sql import SparkSession

# Define a Python function to process comorbidityid
def process_comorbidityid(comorbidityid):
    # Check if comorbidityid contains a comma
    if ',' in comorbidityid:
        # Split comorbidityid by commas
        comorbidity_list = comorbidityid.split(',')
        # Process individual values and map them to the dictionary
        processed_comorbidities = [process_individual_comorbidity(cid) for cid in comorbidity_list]
        # Return the processed values as a comma-separated string
        return ','.join(processed_comorbidities)
    
    # Check if comorbidityid contains a decimal point
    if '.' in comorbidityid:
        # Remove the decimal point
        comorbidityid = comorbidityid.replace('.', '')

    # Look up comorbidityid in the dictionary and return the mapped value if found
    mapped_value = icd_dict.get(comorbidityid, None)
    if mapped_value is not None:
        return mapped_value  # Remove square brackets and return as is

    # If no match found, retain the original comorbidityid
    return comorbidityid

# Define a function to process individual comorbidity values
def process_individual_comorbidity(comorbidity):
    # Remove decimal points
    comorbidity = comorbidity.replace('.', '')
    # Strip square brackets and leading/trailing spaces
    comorbidity = comorbidity.strip('[]').strip()
    # Look up comorbidityid in the dictionary and return the mapped value if found
    mapped_value = icd_dict.get(comorbidity, None)
    if mapped_value is not None:
        return mapped_value
    # If no match found, retain the original comorbidity
    return comorbidity

# Register the process_comorbidityid function as a UDF
process_comorbidityid_udf = udf(process_comorbidityid, StringType())

# Assuming you have a DataFrame named Como_Result_Final1
# Create a new column to store the original comorbidityid values
result_df_full = Como_Result_Final1.withColumn("original_comorbidityid", col("comorbidityid"))

# Apply the UDF to the comorbidityid column and create a new column
result_df_full = result_df_full.withColumn("comorbidityid", process_comorbidityid_udf(col("comorbidityid")))

# Remove square brackets from comorbidityid values
result_df_full = result_df_full.withColumn("comorbidityid", regexp_replace(col("comorbidityid"), r'\[|\]', ''))

# Split the comorbidityid values by comma and explode the resulting array
result_df_full = result_df_full.withColumn("comorbidityid", split(col("comorbidityid"), ",")) \
              .withColumn("comorbidityid", explode(col("comorbidityid")))

# Show the resulting DataFrame
result_df_full.show(100,truncate=False)

+------------------------------------+-------------+----------------------+
|personid                            |comorbidityid|original_comorbidityid|
+------------------------------------+-------------+----------------------+
|0009ad92-088d-4991-82b6-17569e395480|S0993XA      |S09.93XA              |
|002f936d-b037-455e-b331-9980269cc611|H6690        |H66.90                |
|002f936d-b037-455e-b331-9980269cc611|R509         |R50.9                 |
|002f936d-b037-455e-b331-9980269cc611|J40          |J40                   |
|002f936d-b037-455e-b331-9980269cc611|Z20822       |Z20.822               |
|002f936d-b037-455e-b331-9980269cc611|K2970        |K29.70                |
|002f936d-b037-455e-b331-9980269cc611|J020         |J02.0                 |
|002f936d-b037-455e-b331-9980269cc611|S060X0A      |S06.0X0A              |
|002f936d-b037-455e-b331-9980269cc611|X58XXXD      |X58.XXXD              |
|002f936d-b037-455e-b331-9980269cc611|Z4802        |Z48.02                |
|002f936d-b0

In [ ]:
Como_Result_Final1.show(100, truncate=False)

In [ ]:
##################################Final Control Commo Replaced Codes ############################################################
result_df_full.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Normalized_Control_Commo")

In [ ]:
##################################Final Package to be used to normalize commorbidity for cohort group #########################
from pyspark.sql.functions import col, udf, split, regexp_replace, explode
from pyspark.sql.types import StringType

# Define a Python function to process comorbidityid
def process_comorbidityid(comorbidityid):
    # Check if comorbidityid contains a comma
    if ',' in comorbidityid:
        # Split comorbidityid by commas
        comorbidity_list = comorbidityid.split(',')
        # Process individual values and map them to the dictionary
        processed_comorbidities = [process_individual_comorbidity(cid) for cid in comorbidity_list]
        # Return the processed values as a comma-separated string
        return ','.join(processed_comorbidities)
    
    # Check if comorbidityid contains a decimal point
    if '.' in comorbidityid:
        # Remove the decimal point
        comorbidityid = comorbidityid.replace('.', '')

    # Look up comorbidityid in the dictionary and return the mapped value if found
    mapped_value = icd_dict.get(comorbidityid, None)
    if mapped_value is not None:
        return mapped_value  # Remove square brackets and return as is

    # If no match found, retain the original comorbidityid
    return comorbidityid

# Define a function to process individual comorbidity values
def process_individual_comorbidity(comorbidity):
    # Remove decimal points
    comorbidity = comorbidity.replace('.', '')
    # Strip square brackets and leading/trailing spaces
    comorbidity = comorbidity.strip('[]').strip()
    # Look up comorbidityid in the dictionary and return the mapped value if found
    mapped_value = icd_dict.get(comorbidity, None)
    if mapped_value is not None:
        return mapped_value
    # If no match found, retain the original comorbidity
    return comorbidity

# Register the process_comorbidityid function as a UDF
process_comorbidityid_udf = udf(process_comorbidityid, StringType())

# Assuming you have a DataFrame named Como_Result_Final1
# Create a new column to store the original comorbidityid values
result_df_full = Como_Result_Final2.withColumn("original_comorbidityid", col("comorbidityid"))

# Apply the UDF to the comorbidityid column and create a new column
result_df_full = result_df_full.withColumn("comorbidityid", process_comorbidityid_udf(col("comorbidityid")))

# Remove square brackets from comorbidityid values
result_df_full = result_df_full.withColumn("comorbidityid", regexp_replace(col("comorbidityid"), r'\[|\]', ''))

# Split the comorbidityid values by comma and explode the resulting array
result_df_full = result_df_full.withColumn("comorbidityid", split(col("comorbidityid"), ",")) \
              .withColumn("comorbidityid", explode(col("comorbidityid")))

# Show the resulting DataFrame
result_df_full.show(100,truncate=False)

In [ ]:
##################################Final Control Commo Replaced Codes ############################################################
result_df_full.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Normalized_Cohort_Commo")

In [ ]:
total_records = result_df_full.count()
total_records_o = Como_Result_Final1.count()
# Print the total number of records
print(f"Total number of records in result_df: {total_records}")
print(f"Total number of records in result_df: {total_records_o}")

In [ ]:
########################Final Package to be used #####################################################################3
from pyspark.sql.functions import col, udf, split, regexp_replace, explode
from pyspark.sql.types import StringType
from pyspark.sql import SparkSession

# Define a Python function to process comorbidityid
def process_comorbidityid(comorbidityid):
    # Check if comorbidityid contains a comma
    if ',' in comorbidityid:
        # Split comorbidityid by commas
        comorbidity_list = comorbidityid.split(',')
        # Process individual values and map them to the dictionary
        processed_comorbidities = [process_individual_comorbidity(cid) for cid in comorbidity_list]
        # Return the processed values as a comma-separated string
        return ','.join(processed_comorbidities)
    
    # Check if comorbidityid contains a decimal point
    if '.' in comorbidityid:
        # Remove the decimal point
        comorbidityid = comorbidityid.replace('.', '')

    # Look up comorbidityid in the dictionary and return the mapped value if found
    mapped_value = icd_dict.get(comorbidityid, None)
    if mapped_value is not None:
        return mapped_value  # Remove square brackets and return as is

    # If no match found, retain the original comorbidityid
    return comorbidityid

# Define a function to process individual comorbidity values
def process_individual_comorbidity(comorbidity):
    # Remove decimal points
    comorbidity = comorbidity.replace('.', '')
    # Strip square brackets and leading/trailing spaces
    comorbidity = comorbidity.strip('[]').strip()
    # Look up comorbidityid in the dictionary and return the mapped value if found
    mapped_value = icd_dict.get(comorbidity, None)
    if mapped_value is not None:
        return mapped_value
    # If no match found, retain the original comorbidity
    return comorbidity

# Register the process_comorbidityid function as a UDF
process_comorbidityid_udf = udf(process_comorbidityid, StringType())

# Assuming you have a DataFrame named Como_Result_Final1
# Split the comorbidityid values by comma and apply the UDF to each element
result_df_full = Como_Result_Final1.withColumn("comorbidityid", 
    split(col("comorbidityid"), ",").cast(StringType())
).withColumn("comorbidityid", process_comorbidityid_udf(col("comorbidityid")))

# Remove square brackets from comorbidityid values
result_df_full = result_df_full.withColumn("comorbidityid", regexp_replace(col("comorbidityid"), r'\[|\]', ''))
# Split the comorbidityid values by comma and explode the resulting array
result_df_full = result_df_full.withColumn("comorbidityid", split(col("comorbidityid"), ",")) \
              .withColumn("comorbidityid", explode(col("comorbidityid")))
# Show the resulting DataFrame
result_df_full.show(truncate=False)

In [6]:
#Reading-> Commo-Cohort-nm with commas splitted
normalized_cohort = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Final_replacedcodes_addedalldecimals_cohort_commo_RMCM")

In [ ]:
#Reading-> Commo-Control-nm with commas splitted
normalized_control = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Final_replacedcodes_addedalldecimals_control_commo_fullvalRMCM")

In [7]:
#Finalized-Normalized Cohort
#Reading-> Commo-Cohort-nm with commas splitted
Final_normalized_cohort = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Normalized_Cohort_Commo")

In [ ]:
Final_normalized_cohort.show(10, truncate=False)

In [ ]:
from pyspark.sql.functions import col

# Assuming Como_Result_Final2 is your Spark DataFrame
filtered_df_original = Como_Result_Final2.filter(col("personid") == "102f2e46-83d5-489f-8764-eeef64c13e0f")

# Display the filtered DataFrame
filtered_df_original2 = filtered_df_original.count()
print("Count =",filtered_df_original2)
# filtered_df_original.show(20,truncate=False)

In [ ]:
filtered_df_original.show(20,truncate=False)

In [ ]:
from pyspark.sql.functions import col

# Assuming Como_Result_Final2 is your Spark DataFrame
filtered_df_original = Final_normalized_cohort.filter(col("comorbidityid") == "W01198A")

# Display the filtered DataFrame
filtered_df_original2 = filtered_df_original.count()
print("Count =",filtered_df_original2)
filtered_df_original.show(20,truncate=False)

In [ ]:
Final_normalized_cohort.printSchema()

In [8]:
#Reading-> Commo-Control-nm with commas splitted
Final_normalized_control = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Normalized_Control_Commo")

In [12]:
from pyspark.sql.functions import col, count, countDistinct

# Filter the DataFrame to select only rows where "comorbidityid" starts with an alphabet
filtered_df_cohort = Final_normalized_cohort.filter(col("comorbidityid").rlike('^[A-Za-z]'))
# Filter the DataFrame to select only rows where "comorbidityid" starts with an alphabet
filtered_df_control = Final_normalized_control.filter(col("comorbidityid").rlike('^[A-Za-z]'))

# Count both unique and non-unique values of "comorbidityid"
count_df = filtered_df_cohort.groupBy("comorbidityid").agg(count("*").alias("count"))

# Show the unique count and non-unique count
unique_count = count_df.count()
total_count = filtered_df_cohort.count()

print(f"Unique Count of cohort: {unique_count}")
print(f"Total Count of cohort(including duplicates): {total_count}")

# Count both unique and non-unique values of "comorbidityid"
count_df1 = filtered_df_control.groupBy("comorbidityid").agg(count("*").alias("count"))

# Show the unique count and non-unique count
unique_count1 = count_df1.count()
total_count1 = filtered_df_control.count()

print(f"Unique Count of control: {unique_count1}")
print(f"Total Count of control(including duplicates): {total_count1}")

# Combine the comorbidityid values from both DataFrames
combined_df = filtered_df_cohort.union(filtered_df_control)

# Count the unique comorbidityid values
unique_count = combined_df.select("comorbidityid").distinct().count()

# Show the unique count
print(f"Unique Comorbidity Count (Combined): {unique_count}")

Unique Count of cohort: 38034
Total Count of cohort(including duplicates): 7769106
Unique Count of control: 40328
Total Count of control(including duplicates): 19944630
Unique Comorbidity Count (Combined): 46074


In [ ]:
combined_df.printSchema()

In [ ]:
combined_df.show(100, truncate=False)

In [ ]:
# Select distinct comorbidityid values and collect them into a list
unique_comorbidity_list = combined_df.select("comorbidityid").distinct().rdd.map(lambda x: x[0]).collect()
print(unique_comorbidity_list)

In [ ]:
# Filter the DataFrame to select records where "comorbidityid" ends with "XRA"
filtered_df = combined_df.filter(col("comorbidityid").rlike(r'XRA$'))

# Count the number of matching records
count_with_XRA = filtered_df.count()
filtered_df.show()
# Show the count
print(f"Number of Records with 'XRA' in Comorbidity ID: {count_with_XRA}")

In [ ]:
# Assuming you have a DataFrame named combined_df with a "comorbidityid" column

# Filter the DataFrame to exclude "comorbidityid" values that end with "XRA"
filtered_df = combined_df.filter(~col("comorbidityid").rlike(r'XRA$'))

# Select distinct comorbidityid values from the filtered DataFrame
unique_comorbidity_list = filtered_df.select("comorbidityid").distinct().rdd.map(lambda x: x[0]).collect()

# Show the list of unique comorbidityid values (excluding those ending with "XRA")
# print("Unique Comorbidity ID List (Excluding 'XRA' Endings):")
print(unique_comorbidity_list)
# for comorbidityid in unique_comorbidity_list:
#     print(comorbidityid)

In [ ]:
filtered_df.show(100, truncate=False)

In [ ]:
from pyspark.sql.functions import col

# Assuming Como_Result_Final2 is your Spark DataFrame
filtered_df_original2 = filtered_df.filter(col("personid") == "04fbb148-a952-4b9e-9ba8-4e1a51dee61e")

# Display the filtered DataFrame
filtered_df_original2 = filtered_df_original2.count()
print("Count =",filtered_df_original2)

In [10]:
from pyspark.sql.functions import col, monotonically_increasing_id

# Assuming you have a DataFrame named combined_df with a "comorbidityid" column

# Filter the DataFrame to exclude "comorbidityid" values that end with "XRA"
filtered_df = combined_df.filter(~col("comorbidityid").rlike(r'XRA$'))

# Collect all comorbidityid values from the filtered DataFrame into a list
comorbidity_list = filtered_df.select("comorbidityid").rdd.map(lambda x: x[0]).collect()

# Create a DataFrame with the original comorbidityid values
modified_df = spark.createDataFrame([(comorbidityid,) for comorbidityid in comorbidity_list], ["modified_comorbidityid"])

# Add an index column to both DataFrames for joining
filtered_df = filtered_df.withColumn("index", monotonically_increasing_id())
modified_df = modified_df.withColumn("index", monotonically_increasing_id())

# Join the DataFrames on the index column
result_df = filtered_df.join(modified_df, "index").drop("index")

# Show the result DataFrame
result_df.show(50, truncate=False)

# Assuming Como_Result_Final2 is your Spark DataFrame
filtered_df = result_df.filter(col("personid") == "0053f0b8-4a78-4826-b193-9a92cd4e43d5")

# Show the result DataFrame
filtered_df.show(50, truncate=False)

+------------------------------------+-------------+----------------------+----------------------+
|personid                            |comorbidityid|original_comorbidityid|modified_comorbidityid|
+------------------------------------+-------------+----------------------+----------------------+
|41480c39-0394-4c64-be08-3c16bd393c50|M25551       |M25.551               |M25551                |
|5a9d10b3-e59b-4244-bbeb-6e63b892b8e5|R239         |R23.9                 |R239                  |
|0d21b75f-f7e1-4cf9-b8ce-9592f9ca165d|M4802        |M48.02                |M4802                 |
|102f2e46-83d5-489f-8764-eeef64c13e0f|W01198A      |E888.1                |W01198A               |
|73d753f3-d492-47aa-8afe-c0b5aa47fa7f|R51          |R51                   |R51                   |
|7b426cec-1a27-4bdb-81f4-d313b990392e|R569         |780.39                |R569                  |
|77587977-2393-4ca8-a733-541efc50651c|R000         |R00.0                 |R000                  |
|d84a945e-

In [11]:
# Assuming Como_Result_Final2 is your Spark DataFrame
filtered_df = result_df.filter(col("personid") == "77587977-2393-4ca8-a733-541efc50651c")

# Show the result DataFrame
filtered_df.show(50, truncate=False)

+------------------------------------+-------------+----------------------+----------------------+
|personid                            |comorbidityid|original_comorbidityid|modified_comorbidityid|
+------------------------------------+-------------+----------------------+----------------------+
|77587977-2393-4ca8-a733-541efc50651c|R000         |R00.0                 |R000                  |
|77587977-2393-4ca8-a733-541efc50651c|N19          |N19                   |Y92410                |
|77587977-2393-4ca8-a733-541efc50651c|Y929         |E849.9                |J069                  |
|77587977-2393-4ca8-a733-541efc50651c|Z0000        |Z00.00                |Z23                   |
|77587977-2393-4ca8-a733-541efc50651c|W5522XA      |W55.22XA              |X58XXXD               |
|77587977-2393-4ca8-a733-541efc50651c|Z6830        |Z68.30                |Y048XXA               |
|77587977-2393-4ca8-a733-541efc50651c|J1289        |J12.89                |S0093XA               |
|77587977-

In [21]:
from pyspark.sql.functions import col, monotonically_increasing_id
from pyspark.sql.functions import col, regexp_extract

# Assuming you have a DataFrame named combined_df with a "comorbidityid" column

# Filter the DataFrame to exclude "comorbidityid" values that end with "XRA"
filtered_df = combined_df.filter(~col("comorbidityid").rlike(r'XRA$'))

# # # Select distinct comorbidityid values from the filtered DataFrame
# # comorbidity_list = filtered_df.select("comorbidityid").distinct().rdd.map(lambda x: x[0]).collect()

# # Collect all comorbidityid values from the filtered DataFrame into a list
# comorbidity_list = filtered_df.select("comorbidityid").rdd.map(lambda x: x[0]).collect()
# # Create a new list with the modified comorbidityid values
# modified_comorbidity_list = [item[:3] + "." + item[3:] for item in comorbidity_list]
# print(modified_comorbidity_list)

# # Create a DataFrame with the modified comorbidityid values
# modified_df = spark.createDataFrame([(comorbidityid,) for comorbidityid in modified_comorbidity_list], ["modified_comorbidityid"])
# # modified_df.show(truncate=False)

# # # Add an index column to both DataFrames for joining
# # filtered_df = filtered_df.withColumn("index", monotonically_increasing_id())
# # modified_df = modified_df.withColumn("index", monotonically_increasing_id())

# # # Join the DataFrames on the index column
# # result_df = filtered_df.join(modified_df, "index").drop("index")

# # # Show the result DataFrame
# # result_df.show(50, truncate=False)
# # # Assuming Como_Result_Final2 is your Spark DataFrame
# # filtered_df = result_df.filter(col("personid") == "0053f0b8-4a78-4826-b193-9a92cd4e43d5")
# # # Show the result DataFrame
# # filtered_df.show(50, truncate=False)

In [22]:
from pyspark.sql.functions import col, concat, lit, substring

# Add a new column 'modified_Commo' with modified comorbidityid values
modified_df = filtered_df.withColumn(
    "modified_Commo",
    concat(
        substring(col("comorbidityid"), 1, 3), 
        lit("."), 
        substring(col("comorbidityid"), 4, 100)  # Assuming maximum length of comorbidityid is 100 characters
    )
)

# Show the modified DataFrame
modified_df.show(truncate=False)

+------------------------------------+-------------+----------------------+--------------+
|personid                            |comorbidityid|original_comorbidityid|modified_Commo|
+------------------------------------+-------------+----------------------+--------------+
|04fbb148-a952-4b9e-9ba8-4e1a51dee61e|R0600        |R06.00                |R06.00        |
|ff3e10f9-dc5d-4f3c-9196-beba949b287d|T1490        |T14.90                |T14.90        |
|ff3e10f9-dc5d-4f3c-9196-beba949b287d|M25559       |M25.559               |M25.559       |
|cf5d1281-a9f7-44b3-8b39-4037c55363b5|R112         |R11.2                 |R11.2         |
|3f01869e-e267-463b-953d-b48e7c162901|R4182        |R41.82                |R41.82        |
|d558435c-474a-41d2-8489-232fc677aa09|S0990XA      |S09.90XA              |S09.90XA      |
|28bda0b7-89a9-445d-92df-6a2140194be5|M25559       |M25.559               |M25.559       |
|6d05b235-a0d8-4f19-8e16-b25723d89d90|K922         |K92.2                 |K92.2         |

In [23]:
# Assuming Como_Result_Final2 is your Spark DataFrame
result_df = modified_df.filter(col("personid") == "6ce46703-5ca9-42e4-8419-048fd89241db")
# Show the result DataFrame
result_df.show(100, truncate=False)

+------------------------------------+-------------+----------------------+--------------+
|personid                            |comorbidityid|original_comorbidityid|modified_Commo|
+------------------------------------+-------------+----------------------+--------------+
|6ce46703-5ca9-42e4-8419-048fd89241db|G8911        |338.11                |G89.11        |
|6ce46703-5ca9-42e4-8419-048fd89241db|S270XXA      |860.0                 |S27.0XXA      |
|6ce46703-5ca9-42e4-8419-048fd89241db|J948         |511.89                |J94.8         |
|6ce46703-5ca9-42e4-8419-048fd89241db|R1310        |787.20                |R13.10        |
|6ce46703-5ca9-42e4-8419-048fd89241db|S37819A      |868.01                |S37.819A      |
|6ce46703-5ca9-42e4-8419-048fd89241db|W19XXXA      |E888.9                |W19.XXXA      |
|6ce46703-5ca9-42e4-8419-048fd89241db|R41841       |799.52                |R41.841       |
|6ce46703-5ca9-42e4-8419-048fd89241db|S2249XA      |807.09                |S22.49XA      |

In [24]:
from pyspark.sql.functions import col, split


# Modify the "modified_comorbidityid" column to include only values before the floating point
result_df = modified_df.withColumn("modified_Commo", split(col("modified_Commo"), "\\.").getItem(0))

# Show the modified result DataFrame
result_df.show(truncate=False)

+------------------------------------+-------------+----------------------+--------------+
|personid                            |comorbidityid|original_comorbidityid|modified_Commo|
+------------------------------------+-------------+----------------------+--------------+
|04fbb148-a952-4b9e-9ba8-4e1a51dee61e|R0600        |R06.00                |R06           |
|ff3e10f9-dc5d-4f3c-9196-beba949b287d|T1490        |T14.90                |T14           |
|ff3e10f9-dc5d-4f3c-9196-beba949b287d|M25559       |M25.559               |M25           |
|cf5d1281-a9f7-44b3-8b39-4037c55363b5|R112         |R11.2                 |R11           |
|3f01869e-e267-463b-953d-b48e7c162901|R4182        |R41.82                |R41           |
|d558435c-474a-41d2-8489-232fc677aa09|S0990XA      |S09.90XA              |S09           |
|28bda0b7-89a9-445d-92df-6a2140194be5|M25559       |M25.559               |M25           |
|6d05b235-a0d8-4f19-8e16-b25723d89d90|K922         |K92.2                 |K92           |

In [25]:
from pyspark.sql.functions import col, udf, expr, instr

# Define a UDF to extract the part before the floating point
def extract_before_float(s):
    try:
        # Split by '.' and take the first part
        parts = s.split('.', 1)
        return parts[0] if len(parts) > 0 else None
    except ValueError as e:
        # Print additional debugging information
        print(f"Error processing '{s}': {e}")
        return None

# Register the UDF
extract_before_float_udf = udf(extract_before_float)

# Apply the UDF and modify the modified_comorbidityid in-place
combined_addfloatpoint = result_df \
    .withColumn("before_float_modified", extract_before_float_udf(col("modified_Commo"))) \
    .withColumn("dot_position", instr(col("original_comorbidityid"), ".")) \
    .withColumn("after_float_original", expr("substr(original_comorbidityid, dot_position + 1, length(original_comorbidityid))")) \
    .withColumn("Latest_modified_comorbidityid",
                expr("IFNULL(before_float_modified, '') || '.' || IFNULL(after_float_original, '')")) \
    .drop("before_float_modified", "dot_position", "after_float_original")

# Show the result
combined_addfloatpoint.show(truncate=False)

+------------------------------------+-------------+----------------------+--------------+-----------------------------+
|personid                            |comorbidityid|original_comorbidityid|modified_Commo|Latest_modified_comorbidityid|
+------------------------------------+-------------+----------------------+--------------+-----------------------------+
|04fbb148-a952-4b9e-9ba8-4e1a51dee61e|R0600        |R06.00                |R06           |R06.00                       |
|ff3e10f9-dc5d-4f3c-9196-beba949b287d|T1490        |T14.90                |T14           |T14.90                       |
|ff3e10f9-dc5d-4f3c-9196-beba949b287d|M25559       |M25.559               |M25           |M25.559                      |
|cf5d1281-a9f7-44b3-8b39-4037c55363b5|R112         |R11.2                 |R11           |R11.2                        |
|3f01869e-e267-463b-953d-b48e7c162901|R4182        |R41.82                |R41           |R41.82                       |
|d558435c-474a-41d2-8489-232fc67

In [26]:
# Assuming Como_Result_Final2 is your Spark DataFrame
result_df = combined_addfloatpoint.filter(col("personid") == "6ce46703-5ca9-42e4-8419-048fd89241db")
# Show the result DataFrame
result_df.show(100, truncate=False)

+------------------------------------+-------------+----------------------+--------------+-----------------------------+
|personid                            |comorbidityid|original_comorbidityid|modified_Commo|Latest_modified_comorbidityid|
+------------------------------------+-------------+----------------------+--------------+-----------------------------+
|6ce46703-5ca9-42e4-8419-048fd89241db|G8911        |338.11                |G89           |G89.11                       |
|6ce46703-5ca9-42e4-8419-048fd89241db|S270XXA      |860.0                 |S27           |S27.0                        |
|6ce46703-5ca9-42e4-8419-048fd89241db|J948         |511.89                |J94           |J94.89                       |
|6ce46703-5ca9-42e4-8419-048fd89241db|R1310        |787.20                |R13           |R13.20                       |
|6ce46703-5ca9-42e4-8419-048fd89241db|S37819A      |868.01                |S37           |S37.01                       |
|6ce46703-5ca9-42e4-8419-048fd89

In [27]:
from pyspark.sql.functions import col

# Assuming Como_Result_Final2 is your Spark DataFrame
filtered_df = combined_addfloatpoint.filter(col("personid") == "0053f0b8-4a78-4826-b193-9a92cd4e43d5")

# Display the filtered DataFrame
filtered_df_cnt = filtered_df.count()
print("Count =",filtered_df_cnt)

Count = 42


In [28]:
##################################Final Control Commo Replaced Codes ############################################################
combined_addfloatpoint.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Final_Normalized_Commorbidities_tostack")

In [17]:
from pyspark.sql.functions import col

# Assuming Como_Result_Final2 is your Spark DataFrame
filtered_df = result_df.filter(col("personid") == "0053f0b8-4a78-4826-b193-9a92cd4e43d5")

# Display the filtered DataFrame
filtered_df_cnt = filtered_df.count()
print("Count =",filtered_df_cnt)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Count = 42


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
filtered_df.show(42,truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+-------------+----------------------+----------------------+
|personid                            |comorbidityid|original_comorbidityid|modified_comorbidityid|
+------------------------------------+-------------+----------------------+----------------------+
|0053f0b8-4a78-4826-b193-9a92cd4e43d5|M7542        |M75.42                |P08.1                 |
|0053f0b8-4a78-4826-b193-9a92cd4e43d5|F338         |F33.8                 |Z23.                  |
|0053f0b8-4a78-4826-b193-9a92cd4e43d5|F17200       |305.1                 |R45.850               |
|0053f0b8-4a78-4826-b193-9a92cd4e43d5|R1013        |R10.13                |S52.521D              |
|0053f0b8-4a78-4826-b193-9a92cd4e43d5|F411         |F41.1                 |S01.80XD              |
|0053f0b8-4a78-4826-b193-9a92cd4e43d5|M7580        |726.2                 |J01.41                |
|0053f0b8-4a78-4826-b193-9a92cd4e43d5|F3181        |F31.81                |H81.8X9               |
|0053f0b8-

<IPython.core.display.Javascript object>

In [ ]:
from pyspark.sql.functions import col

# Assuming Como_Result_Final2 is your Spark DataFrame
filtered_df_original1 = combined_df.filter(col("personid") == "04fbb148-a952-4b9e-9ba8-4e1a51dee61e")

# Display the filtered DataFrame
filtered_df_original1 = filtered_df_original1.count()
print("Count =",filtered_df_original1)

In [ ]:
# Count the total number of records in the result_df DataFrame
total_records = result_df.count()

# Print the total number of records
print(f"Total number of records in result_df: {total_records}")

In [ ]:
from pyspark.sql.functions import col, split

# Join the DataFrames on the index column
result_df = filtered_df.join(modified_df, "index").drop("index")

# Modify the "modified_comorbidityid" column to include only values before the floating point
result_df = result_df.withColumn("modified_comorbidityid", split(col("modified_comorbidityid"), "\\.").getItem(0))

# Show the modified result DataFrame
result_df.show()

In [ ]:
from pyspark.sql.functions import col

# Assuming Como_Result_Final2 is your Spark DataFrame
filtered_df1 = result_df.filter(col("personid") == "0053f0b8-4a78-4826-b193-9a92cd4e43d5")

# Display the filtered DataFrame
filtered_df_cnt1 = filtered_df1.count()
print("Count =",filtered_df_cnt1)

In [ ]:
from pyspark.sql.functions import col

# Assuming Como_Result_Final2 is your Spark DataFrame
filtered_df_original = result_df.filter(col("personid") == "04fbb148-a952-4b9e-9ba8-4e1a51dee61e")

# Display the filtered DataFrame
filtered_df_original = filtered_df_original.count()
print("Count =",filtered_df_original)

In [ ]:
#write -Finalized Cohort_Commo_Codes_Toprocessfurther
result_df.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/combined_commorbidity_tomapV1")

In [2]:
#Reading
combined_addfloatpoint = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/combined_commorbidity_tomapV1")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [3]:
from pyspark.sql.functions import col

# Assuming Como_Result_Final2 is your Spark DataFrame
filtered_df = combined_addfloatpoint.filter(col("personid") == "0053f0b8-4a78-4826-b193-9a92cd4e43d5")

# Display the filtered DataFrame
filtered_df_cnt = filtered_df.count()
print("Count =",filtered_df_cnt)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Count = 42


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
filtered_df.show(42,truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+-------------+----------------------+----------------------+
|personid                            |comorbidityid|original_comorbidityid|modified_comorbidityid|
+------------------------------------+-------------+----------------------+----------------------+
|0053f0b8-4a78-4826-b193-9a92cd4e43d5|F4310        |F43.10                |R11                   |
|0053f0b8-4a78-4826-b193-9a92cd4e43d5|S46919A      |840.9                 |R51                   |
|0053f0b8-4a78-4826-b193-9a92cd4e43d5|F4312        |F43.12                |E55                   |
|0053f0b8-4a78-4826-b193-9a92cd4e43d5|M7580        |726.2                 |J01                   |
|0053f0b8-4a78-4826-b193-9a92cd4e43d5|F3341        |F33.41                |Y92                   |
|0053f0b8-4a78-4826-b193-9a92cd4e43d5|M7580        |726.19                |M75                   |
|0053f0b8-4a78-4826-b193-9a92cd4e43d5|M7500        |726.0                 |Y92                   |
|0053f0b8-

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from pyspark.sql.functions import col

# Assuming Como_Result_Final2 is your Spark DataFrame
filtered_df_original = combined_addfloatpoint.filter(col("personid") == "04fbb148-a952-4b9e-9ba8-4e1a51dee61e")

# Display the filtered DataFrame
filtered_df_original = filtered_df_original.count()
print("Count =",filtered_df_original)

In [ ]:
from pyspark.sql.functions import col, udf, expr, instr

# Define a UDF to extract the part before the floating point
def extract_before_float(s):
    try:
        # Split by '.' and take the first part
        parts = s.split('.', 1)
        return parts[0] if len(parts) > 0 else None
    except ValueError as e:
        # Print additional debugging information
        print(f"Error processing '{s}': {e}")
        return None

# Register the UDF
extract_before_float_udf = udf(extract_before_float)

# Apply the UDF and modify the modified_comorbidityid in-place
combined_addfloatpoint = combined_addfloatpoint \
    .withColumn("before_float_modified", extract_before_float_udf(col("modified_comorbidityid"))) \
    .withColumn("dot_position", instr(col("original_comorbidityid"), ".")) \
    .withColumn("after_float_original", expr("substr(original_comorbidityid, dot_position + 1, length(original_comorbidityid))")) \
    .withColumn("Latest_modified_comorbidityid",
                expr("IFNULL(before_float_modified, '') || '.' || IFNULL(after_float_original, '')")) \
    .drop("before_float_modified", "dot_position", "after_float_original")

# Show the result
combined_addfloatpoint.show(truncate=False)

In [ ]:
from pyspark.sql import functions as F

# Filter records where Latest_modified_comorbidityid ends with 'XRA$'
filtered_df = combined_addfloatpoint.filter(
    F.col("Latest_modified_comorbidityid").endswith("XRA$")
)

# Count the number of records
count_records = filtered_df.count()

# Print the count
print("Number of records with Latest_modified_comorbidityid ending with 'XRA$':", count_records)

In [ ]:
#Reading
combined_addfloatpoint_Final = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/combined_Commo_addfloatpointV1")

In [ ]:
combined_addfloatpoint.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/combined_Commo_addfloatpointV1")

In [ ]:
Overall_Total = combined_addfloatpoint.count()

# Display the total count
print("Total count of records:", Overall_Total)

In [ ]:
from pyspark.sql.functions import col, coalesce


# Perform a join operation based on original_comorbidityid
joined_df = Final_normalized_cohort \
    .join(result_df, "original_comorbidityid", "left_outer") \
    .select(
        Final_normalized_cohort["personid"],
        Final_normalized_cohort["comorbidityid"],
        Final_normalized_cohort["original_comorbidityid"],
        col("modified_comorbidityid").alias("fn_modified_comorbidityid"),
        result_df["modified_comorbidityid"].alias("result_modified_comorbidityid")
    )

# Add the modified_comorbidityid column to the Final_normalized_cohort DataFrame
Final_result_cohort = joined_df \
    .withColumn(
        "modified_comorbidityid",
        coalesce(
            col("fn_modified_comorbidityid"),
            col("result_modified_comorbidityid")
        )
    ) \
    .drop("fn_modified_comorbidityid", "result_modified_comorbidityid")

# Show the resulting DataFrame
Final_result_cohort.show()

In [ ]:
# Assuming you have a DataFrame named combined_df with a "comorbidityid" column

# Filter the DataFrame to exclude "comorbidityid" values that end with "XRA"
filtered_df = combined_df.filter(~col("comorbidityid").rlike(r'XRA$'))

# Select distinct comorbidityid values from the filtered DataFrame
unique_comorbidity_list = filtered_df.select("comorbidityid").distinct().rdd.map(lambda x: x[0]).collect()

# Create a new list with the modified comorbidityid values
modified_comorbidity_list = [item[:3] + "." + item[3:] for item in unique_comorbidity_list]
print(modified_comorbidity_list)

# # Show the list of unique comorbidityid values with floating points (excluding those ending with "XRA")
# print("Unique Comorbidity ID List with Floating Points (Excluding 'XRA' Endings):")
# for comorbidityid in modified_comorbidity_list:
#     print(comorbidityid)

In [ ]:
# Count the number of items in the list
number_of_items = len(modified_comorbidity_list)

# Print the result
print(f'The number of items in modified_comorbidity_list is: {number_of_items}')

In [ ]:
# Assuming you have a modified_comorbidity_list with values like "123.456" and you want to keep only "123"

# Create a new list with values before the floating point
new_list = [item.split(".")[0] for item in modified_comorbidity_list]
print(new_list)

# # Show the list of modified comorbidityid values with only values before the floating point
# print("Modified Comorbidity ID List with Values Before the Floating Point:")
# for comorbidityid in new_list:
#     print(comorbidityid)

In [ ]:
# Use set to get unique values
unique_values = set(new_list)

# Print unique values
print("Unique Values:")
for value in unique_values:
    print(value)

# Print the total number of unique values
total_unique_count = len(unique_values)
print("\nTotal Number of Unique Values:", total_unique_count)

In [25]:
##################################Final Control Commo Replaced Codes ############################################################
#Reading
combined_addfloatpoint_Final = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Final_Normalized_Commorbidities_tostack")

In [26]:
combined_addfloatpoint_Final.printSchema()

root
 |-- personid: string (nullable = true)
 |-- comorbidityid: string (nullable = true)
 |-- original_comorbidityid: string (nullable = true)
 |-- modified_Commo: string (nullable = true)
 |-- Latest_modified_comorbidityid: string (nullable = true)



In [27]:
# Assuming you have a DataFrame named "df" with the column you want to rename
combined_addfloatpoint_Final = combined_addfloatpoint_Final.withColumnRenamed("comorbidityid", "comorbidityid_RF")
combined_addfloatpoint_Final = combined_addfloatpoint_Final.withColumnRenamed("original_comorbidityid", "comorbidityid")
combined_addfloatpoint_Final.printSchema()

root
 |-- personid: string (nullable = true)
 |-- comorbidityid_RF: string (nullable = true)
 |-- comorbidityid: string (nullable = true)
 |-- modified_Commo: string (nullable = true)
 |-- Latest_modified_comorbidityid: string (nullable = true)



In [28]:
#Reading-> Commo-Cohort
combined_addfloatpoint_Final = combined_addfloatpoint_Final.distinct()

In [29]:
combined_addfloatpoint_cnt = combined_addfloatpoint_Final.count()
print("combined_addfloatpoint_cnt", combined_addfloatpoint_cnt)

combined_addfloatpoint_cnt 27713703


In [30]:
from pyspark.sql.functions import col
filtered_df = combined_addfloatpoint_Final.filter(col("comorbidityid").rlike(",+"))
filtered_df_cnt1 = filtered_df.count()
print("Count of cohort =",filtered_df_cnt1)
# Show the resulting dataframe
filtered_df.show(truncate=False)

Count of cohort = 9467
+------------------------------------+----------------+--------------------+--------------+-----------------------------+
|personid                            |comorbidityid_RF|comorbidityid       |modified_Commo|Latest_modified_comorbidityid|
+------------------------------------+----------------+--------------------+--------------+-----------------------------+
|02238865-3f76-4ae0-a85b-758c4761d827|I8312           |I83.11, I83.12      |I83           |I83.11, I83.12               |
|f9108842-952f-4a17-8ed5-1fe63dc06902|E0869           |E08.69, Z79.4       |E08           |E08.69, Z79.4                |
|8e484c43-4593-4381-bd22-264a0199e170|S069X9A         |S02.119A, S06.9X9A  |S06           |S06.119A, S06.9X9A           |
|7ab9e690-6436-4231-91fd-a8312b629403|I2510           |I25.10, I25.83      |I25           |I25.10, I25.83               |
|7ab9e690-6436-4231-91fd-a8312b629403|M79605          |M79.604, M79.605    |M79           |M79.604, M79.605             |
|

In [6]:
# from pyspark.sql.functions import split, explode

# # Assuming Como_Result_Final2 is your Spark DataFrame
# # Splitting the comorbidityid values by commas and exploding them into separate records
# split_df = Como_Result_Final2.withColumn("comorbidityid", explode(split(col("comorbidityid"), ",")))

# # Show the resulting dataframe
# split_df.show()
from pyspark.sql.functions import split, explode, trim, col

# Assuming Como_Result_Final2 is your Spark DataFrame
# Splitting the comorbidityid values by commas and exploding them into separate records
split_df_CTL = Como_Result_Final1.withColumn("comorbidityid", explode(split(col("comorbidityid"), ",")))

# Remove leading and trailing whitespace
split_df_CTL = split_df_CTL.withColumn("comorbidityid", trim(col("comorbidityid")))

# Show the resulting dataframe
split_df_CTL.show()

+--------------------+-------------+
|            personid|comorbidityid|
+--------------------+-------------+
|0002ff17-107a-4ef...|        L85.3|
|0011202e-a11e-4f1...|     V89.2XXA|
|0016279d-6d04-415...|        Y99.0|
|002dd2ba-6826-4ea...|          R05|
|00345ebc-b703-4f6...|       G93.49|
|0037381b-90b3-4d8...|       R11.10|
|0038904d-c2de-48c...|     S62.617A|
|003b691e-e7b4-4f9...|        Z88.8|
|0040086c-2100-40a...|     S80.812A|
|00492c4b-c094-493...|        583.9|
|004c4c4c-2073-479...|      O99.810|
|00585ef4-c94d-46f...|       V58.66|
|005d1172-1893-402...|     S01.01XA|
|005f205e-6890-459...|       G93.40|
|005f4882-ff26-4ed...|       Z68.25|
|0067af98-437e-438...|     T76.92XA|
|0067b175-8b70-407...|      H93.299|
|0078211f-e93f-4e0...|     V00.131A|
|0086b000-559e-480...|      M25.562|
|0088b23a-9bb7-42e...|     S09.90XA|
+--------------------+-------------+
only showing top 20 rows



In [8]:
from pyspark.sql.functions import col
# Check for duplicates
duplicateCount = Como_Result_Final1 \
    .groupBy("personid", "comorbidityid") \
    .count() \
    .filter(col("count") > 1) \
    .count()

# Print the result
if duplicateCount > 0:
    print("There are duplicate combinations of personid and comorbidityid.")
else:
    print("There are no duplicate combinations of personid and comorbidityid.")

There are no duplicate combinations of personid and comorbidityid.


In [15]:
from pyspark.sql.functions import col

# Check for duplicates
duplicateCount = split_df_CTL \
    .groupBy("personid", "comorbidityid") \
    .count()

# Count the number of rows where the count is greater than 1
duplicateCount1 = duplicateCount.filter(col("count") > 1).count()

# Print the result
if duplicateCount1 > 1:
    print("There are duplicate combinations of personid and comorbidityid.")
else:
    print("There are no duplicate combinations of personid and comorbidityid.")

# Show the duplicate count DataFrame
duplicateCount.show(truncate=False)

# Print the count of duplicate combinations
print("Number of duplicate combinations:", duplicateCount1)

There are duplicate combinations of personid and comorbidityid.
+------------------------------------+-------------+-----+
|personid                            |comorbidityid|count|
+------------------------------------+-------------+-----+
|0002ff17-107a-4ef8-b724-8057c524dfa5|L85.3        |1    |
|0011202e-a11e-4f17-a9e7-5a03b0f47bb7|V89.2XXA     |1    |
|0016279d-6d04-4159-9f7e-f4a4531cdfa4|Y99.0        |1    |
|002dd2ba-6826-4ea4-9c98-fb1a0f906153|R05          |1    |
|00345ebc-b703-4f63-8891-12ae75a0a239|G93.49       |1    |
|0037381b-90b3-4d8a-8b85-251e62e2d564|R11.10       |1    |
|0038904d-c2de-48cc-8e65-019acd1c1aa9|S62.617A     |1    |
|003b691e-e7b4-4f95-bb31-46174c2f24b6|Z88.8        |1    |
|0040086c-2100-40a2-9108-450a75f56f0e|S80.812A     |1    |
|00492c4b-c094-4933-afb1-0a3ecb0a5361|583.9        |1    |
|004c4c4c-2073-4791-8303-8dffa6a3ded9|O99.810      |1    |
|00585ef4-c94d-46f1-b78a-1bbf0249c3be|V58.66       |1    |
|005d1172-1893-4021-97ad-1b4268c85ba3|S01.01XA     

In [7]:
print(split_df_CTL.count())
D_split_df_CTL = split_df_CTL.distinct()
print(D_split_df_CTL.count())

20502939
20499110


In [8]:
# from pyspark.sql.functions import col

# # Assuming Como_Result_Final2 is your Spark DataFrame
# filtered_df = split_df.filter(col("comorbidityid").rlike(",+"))
# filtered_df_cnt1 = filtered_df.count()
# print("Count of cohort =",filtered_df_cnt1)
# # Show the resulting dataframe
# filtered_df.show(truncate=False)
from pyspark.sql.functions import col

# Assuming Como_Result_Final2 is your Spark DataFrame
filtered_df = split_df_CTL.filter(col("comorbidityid").rlike(",+"))
filtered_df_cnt1 = filtered_df.count()
print("Count of control =",filtered_df_cnt1)
# Show the resulting dataframe
filtered_df.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+--------+-------------+
|personid|comorbidityid|
+--------+-------------+
+--------+-------------+



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
BeforeSplit_cnt = Como_Result_Final1.count()
print("Control_Paired_cnt", BeforeSplit_cnt)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Control_Paired_cnt 20499629


<IPython.core.display.Javascript object>

In [10]:
Como_Result_Final1_dt = Como_Result_Final1.distinct()

▸,:,


In [11]:
BeforeSplit_cnt_dt = Como_Result_Final1_dt.count()
print("Control_Paired_cnt", BeforeSplit_cnt_dt)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Control_Paired_cnt 20499629


<IPython.core.display.Javascript object>

In [12]:
AfterSplit_cnt = split_df_CTL.count()
print("Control_Paired_cnt", AfterSplit_cnt)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Control_Paired_cnt 20502939


<IPython.core.display.Javascript object>

In [13]:
Control_AfterSplit_dt = split_df_CTL.distinct()

▸,:,


In [14]:
AfterSplit_cnt_dt = Control_AfterSplit_dt.count()
print("Control_Paired_cnt", AfterSplit_cnt_dt)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Control_Paired_cnt 20499110


<IPython.core.display.Javascript object>

In [151]:
##################################Final Control Commo Replaced Codes ############################################################
Cohort_AfterSplit_dt.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Cohort_Paired_Commo_RemComma")

In [167]:
##################################Final Control Commo Replaced Codes ############################################################
Control_AfterSplit_dt.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Control_Paired_Commo_RemComma")

In [6]:
#Reading-> Commo-Control
Cohort_Paired = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Cohort_Paired_Commo_RemComma")

In [39]:
from pyspark.sql.functions import col

# Alias the personid column from Como_Result_Final2
Cohort_Paired = Cohort_Paired.withColumnRenamed("personid", "Como_personid")
Cohort_Paired = Cohort_Paired.withColumnRenamed("comorbidityid", "Como_comorbidityid")
# Perform left outer join
joined_df = Cohort_Paired.join(
    combined_addfloatpoint_Final,
    (Cohort_Paired.Como_personid == combined_addfloatpoint_Final.personid) &
    (Cohort_Paired.Como_comorbidityid == combined_addfloatpoint_Final.comorbidityid),
    "left_outer"
)

# Selecting all columns from combined_addfloatpoint and comorbidityid column from Como_Result_Final2
final_df = joined_df.select(
    Cohort_Paired.Como_personid.alias("personid"),  # Alias personid column
    Cohort_Paired.Como_comorbidityid.alias("comorbidityid"),  # Alias comorbidityid column from combined_addfloatpoint_Final
    combined_addfloatpoint_Final.comorbidityid_RF,
    combined_addfloatpoint_Final.modified_Commo,
    combined_addfloatpoint_Final.Latest_modified_comorbidityid
)
# Remove duplicate records
final_df = final_df.distinct()

# Show the resulting dataframe
final_df.show(20, truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+-------------+----------------+--------------+-----------------------------+
|personid                            |comorbidityid|comorbidityid_RF|modified_Commo|Latest_modified_comorbidityid|
+------------------------------------+-------------+----------------+--------------+-----------------------------+
|001e4203-1f47-4028-9aad-26a1ae3c9c5e|959.1        |null            |null          |null                         |
|003e8f9f-ef50-4ff2-813c-1ff26464c90c|Y90.7        |Y907            |Y90           |Y90.7                        |
|004eb15d-2644-4385-a5df-e23f7950720e|16932000     |null            |null          |null                         |
|00743d19-e9a9-45d0-bd7e-33799dda882c|446367003    |null            |null          |null                         |
|008a11f8-2bc1-4c84-8351-c78158e037ae|079.99       |null            |null          |null                         |
|008cb221-7b80-4018-9736-622e8909db5b|49727002     |null            |null       

<IPython.core.display.Javascript object>

In [42]:
# Assuming final_df is your Spark DataFrame
filtered_df_cnt1 = final_df.count()
# Display the filtered DataFrame
filtered_df_cnt2 = split_df.count()
print("Count =",filtered_df_cnt1)
print("Count =",filtered_df_cnt2)
# # Show the resulting distinct dataframe
# distinct_df.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Count = 7929539
Count = 7931635


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [43]:
##################################Final Control Commo Replaced Codes ############################################################
final_df.write.mode("overwrite").parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Final_Commo_Cohort_M1")

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
#Reading-> Commo-Control
Cohort_Paired = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Cohort_Paired_Commo_RemComma")

In [5]:
#Reading-> Commo-Control
Final_DF = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/Final_Commo_Cohort_M1")

In [32]:
Cnt1 = Cohort_Paired.count()
print("Count =",Cnt1)
Cnt2 = Final_DF.count()
print("Count =",Cnt2)

▸,:,


<IPython.core.display.Javascript object>

Count = 7931635


<IPython.core.display.Javascript object>

Count = 7929539


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
# Count the number of records in each dataframe
final_df_count = Final_DF.count()
cohort_paired_count = Cohort_Paired.count()

# Check for the difference in counts
if final_df_count != cohort_paired_count:
    print("Records count differs between Final_DF and Cohort_Paired:")
    print("Final_DF count:", final_df_count)
    print("Cohort_Paired count:", cohort_paired_count)
    
    # Perform full outer join
    full_outer_df = Final_DF.join(Cohort_Paired, Final_DF.personid == Cohort_Paired.personid, "full_outer")
    
    # Filter for records where one side is null
    differing_records = full_outer_df.filter((Final_DF.personid.isNull()) | (Cohort_Paired.personid.isNull()))
    
    # Show the differing records
    differing_records.show()
else:
    print("Records count is same between Final_DF and Cohort_Paired:", final_df_count)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+--------+-------------+----------------+--------------+-----------------------------+--------+-------------+
|personid|comorbidityid|comorbidityid_RF|modified_Commo|Latest_modified_comorbidityid|personid|comorbidityid|
+--------+-------------+----------------+--------------+-----------------------------+--------+-------------+
+--------+-------------+----------------+--------------+-----------------------------+--------+-------------+



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
# Filter for null values in the personid column of Final_DF and Cohort_Paired
final_df_null = Final_DF.filter(Final_DF.personid.isNull())
cohort_paired_null = Cohort_Paired.filter(Cohort_Paired.personid.isNull())

# Show the resulting DataFrames
print("Null values in Final_DF:")
final_df_null.show()
print("Null values in Cohort_Paired:")
cohort_paired_null.show()

▸,:,


Null values in Final_DF:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+--------+-------------+
|personid|comorbidityid|
+--------+-------------+
+--------+-------------+



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [6]:
# Check for duplicate personid values in Final_DF and Cohort_Paired
final_df_duplicates = Final_DF.groupBy("personid").count().where("count > 1")
cohort_paired_duplicates = Cohort_Paired.groupBy("personid").count().where("count > 1")

# Show the resulting DataFrames
print("Duplicate personid values in Final_DF:")
final_df_duplicates.show(truncate=False)
print("Duplicate personid values in Cohort_Paired:")
cohort_paired_duplicates.show(truncate=False)

▸,:,


Duplicate personid values in Final_DF:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+-----+
|personid                            |count|
+------------------------------------+-----+
|bcf04052-57c2-485e-970d-16f84d20eb82|253  |
|c873ab9c-f3d7-4794-aa13-0328ec43707c|117  |
|14898267-d439-4815-ac76-90458a31dd39|139  |
|655354b4-6ba7-4593-a005-17d0c8782193|92   |
|e62009b0-66fc-4503-8147-e56ac7314d43|99   |
|4bca8e76-1ae8-412f-892f-9cd0140777ac|52   |
|2a6145c2-b763-48f7-89c6-bd18a726a7b1|79   |
|60b3ae89-7e0e-4046-ac2f-c62c719a6926|34   |
|398f2fea-e43d-4827-adb8-b47d1d555a5b|67   |
|614be368-5bd7-4d02-95c0-f2ba03fbd7fe|87   |
|2c26b328-74b2-464b-8ad8-222b4bba6db4|85   |
|3132a6ce-38ad-45fb-a48d-f9438f66249c|143  |
|d384ef02-20eb-4751-99a4-59b63240bae9|99   |
|69d22b58-4ce7-444e-b038-3224d0be5838|76   |
|0e6122ea-6a03-450a-b41f-6237da2669c4|15   |
|d12e6211-2c57-4d3d-8990-5d4ad884bf03|148  |
|44cc2d70-7132-44bb-b30c-0234fee612a1|3    |
|936f9633-7535-44f7-b70c-8ca3ab63fa4d|99   |
|530aab5e-1e58-4e2e-bcdf-41feda7bf54d|27   |
|de294b83-

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+-----+
|personid                            |count|
+------------------------------------+-----+
|52414c71-46b4-49d2-882f-70f74a646782|82   |
|c9328d3e-366e-4d43-b432-f06c9926ec93|129  |
|99e81e69-8206-4e63-a315-7c85dcea661b|62   |
|733e61b8-bf78-4590-8283-8149a9893bb0|56   |
|92dc838a-09ce-4f69-b16d-fc84d029368b|78   |
|f249c5e4-1ffd-4dbf-86b8-fc40f61c0339|70   |
|a6ddbabb-d5fe-4296-b7b4-04decdbf173b|68   |
|39b9bf74-f969-45d6-badf-85cf75b712d9|56   |
|3b52cf05-a9bc-4588-b105-0e2d7b3677ce|103  |
|b55031c9-5d23-4377-9d46-d2ca47f3a467|122  |
|fdcbbd5a-4710-4ea4-b6c3-21ee1c71ee1b|45   |
|a9df2d41-4c2e-40ef-b09e-94ca17f09933|97   |
|6d9759c3-4f86-45e3-b3ce-520b340aa9ab|117  |
|a39e497b-06dd-4fc7-9c5f-f38858ef7b12|65   |
|277bfef4-f824-49ab-982e-9eec41b3ecf9|66   |
|2a248500-b85b-4c45-8c40-d1ea73fae055|19   |
|a4922c7e-d535-479a-96c4-2796898c4a83|181  |
|8a553e36-569a-4494-870e-d83a7ba47b1a|98   |
|c3a8b715-466d-4119-9cb0-5d83b868b1c5|64   |
|b8d511f4-

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
# Find records associated with duplicate personid values in Final_DF
final_df_duplicate_records = Final_DF.join(final_df_duplicates.select("personid"), "personid", "inner")

# Find records associated with duplicate personid values in Cohort_Paired
cohort_paired_duplicate_records = Cohort_Paired.join(cohort_paired_duplicates.select("personid"), "personid", "inner")

# Show the resulting DataFrames
print("Records associated with duplicate personid values in Final_DF:")
final_df_duplicate_records.show(truncate=False)
print("Records associated with duplicate personid values in Cohort_Paired:")
cohort_paired_duplicate_records.show(truncate=False)

▸,:,


Records associated with duplicate personid values in Final_DF:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+-------------+
|personid                            |comorbidityid|
+------------------------------------+-------------+
|0053f0b8-4a78-4826-b193-9a92cd4e43d5|719.41       |
|0053f0b8-4a78-4826-b193-9a92cd4e43d5|91019004     |
|0053f0b8-4a78-4826-b193-9a92cd4e43d5|726.19       |
|0053f0b8-4a78-4826-b193-9a92cd4e43d5|F60.3        |
|0053f0b8-4a78-4826-b193-9a92cd4e43d5|S46.912A     |
|0053f0b8-4a78-4826-b193-9a92cd4e43d5|F43.12       |
|0053f0b8-4a78-4826-b193-9a92cd4e43d5|466.0        |
|0053f0b8-4a78-4826-b193-9a92cd4e43d5|296.30       |
|0053f0b8-4a78-4826-b193-9a92cd4e43d5|25064002     |
|0053f0b8-4a78-4826-b193-9a92cd4e43d5|840.9        |
|0053f0b8-4a78-4826-b193-9a92cd4e43d5|F33.0        |
|0053f0b8-4a78-4826-b193-9a92cd4e43d5|726.10       |
|0053f0b8-4a78-4826-b193-9a92cd4e43d5|M75.52       |
|0053f0b8-4a78-4826-b193-9a92cd4e43d5|309.81       |
|0053f0b8-4a78-4826-b193-9a92cd4e43d5|F33.2        |
|0053f0b8-4a78-4826-b193-9a92cd4e43d5|784.0   

<IPython.core.display.Javascript object>

In [8]:
# Filter Final_DF for the specific personid and display comorbidityid values
final_df_specific_personid = Final_DF.filter(Final_DF.personid == "0053f0b8-4a78-4826-b193-9a92cd4e43d5")
print("Comorbidityid values in Final_DF for the specific personid:")
final_df_specific_personid.select("comorbidityid").distinct().show(truncate=False)

# Filter Cohort_Paired for the specific personid and display comorbidityid values
cohort_paired_specific_personid = Cohort_Paired.filter(Cohort_Paired.personid == "0053f0b8-4a78-4826-b193-9a92cd4e43d5")
print("Comorbidityid values in Cohort_Paired for the specific personid:")
cohort_paired_specific_personid.select("comorbidityid").distinct().show(truncate=False)

▸,:,


Comorbidityid values in Final_DF for the specific personid:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+-------------+
|comorbidityid|
+-------------+
|466.0        |
|309.81       |
|784.0        |
|F33.2        |
|R07.9        |
|726.19       |
|J44.9        |
|R10.13       |
|Z20.828      |
|F43.10       |
|726.2        |
|M75.52       |
|F41.0        |
|Z86.19       |
|719.41       |
|296.30       |
|305.1        |
|25064002     |
|569.3        |
|F31.81       |
+-------------+
only showing top 20 rows

Comorbidityid values in Cohort_Paired for the specific personid:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+-------------+
|comorbidityid|
+-------------+
|466.0        |
|309.81       |
|784.0        |
|F33.2        |
|726.19       |
|R07.9        |
|J44.9        |
|R10.13       |
|Z20.828      |
|F43.10       |
|726.2        |
|M75.52       |
|F41.0        |
|Z86.19       |
|719.41       |
|296.30       |
|305.1        |
|25064002     |
|569.3        |
|F31.81       |
+-------------+
only showing top 20 rows



<IPython.core.display.Javascript object>

In [15]:
# Select only the common columns in both DataFrames
final_df_selected = Final_DF.select("personid", "comorbidityid")
record_ct1 = Final_DF.count()
record_ct2 = Cohort_Paired.count()
# Calculate the difference in counts
count_difference = abs(record_ct1 - record_ct2)
# Display the difference in counts
print("Difference in record counts:", count_difference)
# Perform the exceptAll operation
record_differences = final_df_selected.exceptAll(Cohort_Paired)
cnt = record_differences.count()
print(cnt)
# Display the record differences
record_differences.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+-------------+
|personid                            |comorbidityid|
+------------------------------------+-------------+
|4c39f02a-8228-49f0-8db3-6f979930d88a|H02.831      |
|80284619-a328-477b-b3b8-933e5606a2c3|M25.851      |
|9467407a-6fcd-4a91-b800-d3bc5b57f240|H52.03       |
|bb968f99-d727-4a06-b7ae-422e5fea0576|M25.561      |
|d28135e4-49b9-456c-a1f2-73b1f23663c7|B49          |
|2ed139b7-6c08-4bb2-a644-4f0baed7d41c|H04.129      |
|483d602a-7062-41f1-b3a7-3eebc9baca88|M25.562      |
|67428339-648b-427e-9709-50285cad75fb|R65.20       |
|b86f7666-9996-434e-a000-5e5327c45d83|I15.1        |
|dcad30ab-166a-4dd3-9b1a-9ba895dd0898|R20.2        |
|ddc55306-31ee-4a83-a74b-8d630edf6ce9|H01.02A      |
|fe0e49cc-eb80-4ea1-b987-644a81ceb65c|M25.561      |
|08dac490-eb03-48e1-a496-7fc16d31e757|W19.XXXA     |
|3abdb285-0fc3-4284-846a-6b14d83307ea|H02.88B      |
|44456354-5362-4afa-9ae7-64c8dd8f52d2|Z68.38       |
|54c37e68-fc19-48de-97e4-77a89cc1e5fd|M54.6   

<IPython.core.display.Javascript object>

In [18]:
from pyspark.sql.functions import lit

# Select only the common columns in both DataFrames
final_df_selected = Final_DF.select("personid", "comorbidityid")
cohort_paired_selected = Cohort_Paired.select("personid", "comorbidityid")

# Ensure both DataFrames have the same schema
final_df_selected = final_df_selected.withColumn("dummy", lit(None))
cohort_paired_selected = cohort_paired_selected.withColumn("dummy", lit(None))

# Perform the exceptAll operation
record_differences = final_df_selected.exceptAll(cohort_paired_selected)

# Calculate the difference in counts
cnt = record_differences.count()
print("record_differences",cnt)
record_ct1 = Final_DF.count()
record_ct2 = Cohort_Paired.count()
count_difference = abs(record_ct1 - record_ct2)

# Display the difference in counts
print("Difference in record counts:", count_difference)

# Display the record differences
record_differences.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+-------------+-----+
|personid                            |comorbidityid|dummy|
+------------------------------------+-------------+-----+
|4c39f02a-8228-49f0-8db3-6f979930d88a|H02.831      |null |
|80284619-a328-477b-b3b8-933e5606a2c3|M25.851      |null |
|9467407a-6fcd-4a91-b800-d3bc5b57f240|H52.03       |null |
|bb968f99-d727-4a06-b7ae-422e5fea0576|M25.561      |null |
|d28135e4-49b9-456c-a1f2-73b1f23663c7|B49          |null |
|2ed139b7-6c08-4bb2-a644-4f0baed7d41c|H04.129      |null |
|483d602a-7062-41f1-b3a7-3eebc9baca88|M25.562      |null |
|67428339-648b-427e-9709-50285cad75fb|R65.20       |null |
|b86f7666-9996-434e-a000-5e5327c45d83|I15.1        |null |
|dcad30ab-166a-4dd3-9b1a-9ba895dd0898|R20.2        |null |
|ddc55306-31ee-4a83-a74b-8d630edf6ce9|H01.02A      |null |
|fe0e49cc-eb80-4ea1-b987-644a81ceb65c|M25.561      |null |
|08dac490-eb03-48e1-a496-7fc16d31e757|W19.XXXA     |null |
|3abdb285-0fc3-4284-846a-6b14d83307ea|H02.88B      |null

<IPython.core.display.Javascript object>

In [25]:
# Filter records based on conditions
filtered_records = Final_DF.filter((Final_DF["personid"] == "4c39f02a-8228-49f0-8db3-6f979930d88a") & (Final_DF["comorbidityid"] == "H02.831"))

# Show the filtered records
filtered_records.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+-------------+----------------+--------------+-----------------------------+
|personid                            |comorbidityid|comorbidityid_RF|modified_Commo|Latest_modified_comorbidityid|
+------------------------------------+-------------+----------------+--------------+-----------------------------+
|4c39f02a-8228-49f0-8db3-6f979930d88a|H02.831      |null            |null          |null                         |
+------------------------------------+-------------+----------------+--------------+-----------------------------+



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [41]:
filtered_df1.show(160, truncate=False)
filtered_df2.show(157, truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+-------------+----------------+--------------+-----------------------------+
|personid                            |comorbidityid|comorbidityid_RF|modified_Commo|Latest_modified_comorbidityid|
+------------------------------------+-------------+----------------+--------------+-----------------------------+
|483d602a-7062-41f1-b3a7-3eebc9baca88|M25.562      |null            |null          |null                         |
|483d602a-7062-41f1-b3a7-3eebc9baca88|T50.905A     |T50905A         |T50           |T50.905A                     |
|483d602a-7062-41f1-b3a7-3eebc9baca88|M47.896      |M47896          |M47           |M47.896                      |
|483d602a-7062-41f1-b3a7-3eebc9baca88|M54.50       |null            |null          |null                         |
|483d602a-7062-41f1-b3a7-3eebc9baca88|M46.96       |M4696           |M46           |M46.96                       |
|483d602a-7062-41f1-b3a7-3eebc9baca88|S32.009K     |S32009K         |S32        

<IPython.core.display.Javascript object>

In [11]:
from pyspark.sql.functions import col
# Display the filtered DataFrame
filtered_df_cnt1 = Como_Result_Final2.count()
print("Count of cohort =",filtered_df_cnt1)
# Display the filtered DataFrame
filtered_df_cnt2 = final_df.count()
print("Count of cohort after processing and mapping done=",filtered_df_cnt2)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Count of cohort = 7930014


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Count of cohort after processing and mapping done= 7931635


<IPython.core.display.Javascript object>

In [13]:
from pyspark.sql.functions import col

# Assuming Como_Result_Final2 is your Spark DataFrame
filtered_df = Como_Result_Final2.filter(col("comorbidityid").rlike(",+"))
filtered_df_cnt1 = filtered_df.count()
print("Count of cohort =",filtered_df_cnt1)
# Show the resulting dataframe
filtered_df.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Count of cohort = 1469


<IPython.core.display.Javascript object>

+------------------------------------+----------------------+
|Como_personid                       |Como_comorbidityid    |
+------------------------------------+----------------------+
|483d602a-7062-41f1-b3a7-3eebc9baca88|M18.11, M18.12        |
|eb23204c-5678-4740-8c86-af15047753bb|I67.9, G81.14         |
|600eeedf-529c-4597-a02b-ad3fa2d89334|N39.0, R31.9          |
|1ffec02d-67ec-4bf0-b078-d96bc9187d42|M54.42, G89.29        |
|7775eca6-5e3c-409e-a4a8-915f91962eb5|S40.861A, W57.XXXA    |
|658c8258-183f-4f7b-b4bc-882a55293f7d|S31.139A, W34.00XA    |
|fad0a856-4b52-4123-8d17-3623f0fa0f6b|M25.521, M25.522      |
|849280da-0529-480c-a4ac-11a467d7fd87|E11.42, Z79.4         |
|5165b023-1129-481b-9057-c008002a1295|S01.83XA, W34.00XA    |
|722ec747-915b-465d-8e67-81952a65901a|E10.29, R80.9         |
|a05ce33e-2417-4eff-bd64-dffca1f24154|O99.282, E03.9        |
|c527ce85-5f40-4568-ae8d-c43c99b3ea4d|I15.1, N28.89         |
|40e5e09a-15a3-437a-9012-dd5ebefd4ac8|Z98.890, Z95.828      |
|83b75dc

<IPython.core.display.Javascript object>

In [36]:
from pyspark.sql.functions import col

# Assuming Como_Result_Final2 is your Spark DataFrame
filtered_df1 = split_df.filter(col("personid") == "483d602a-7062-41f1-b3a7-3eebc9baca88")

# Display the filtered DataFrame
filtered_df_cnt1 = filtered_df1.count()
print("Count =",filtered_df_cnt1)
# Assuming Como_Result_Final2 is your Spark DataFrame
filtered_df2 = Como_Result_Final2.filter(col("personid") == "483d602a-7062-41f1-b3a7-3eebc9baca88")

# Display the filtered DataFrame
filtered_df_cnt2 = filtered_df2.count()
print("Count =",filtered_df_cnt2)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Count = 171


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Count = 160


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [33]:
filtered_df2.show(160,truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+------------------------+
|personid                            |comorbidityid           |
+------------------------------------+------------------------+
|483d602a-7062-41f1-b3a7-3eebc9baca88|M18.11, M18.12          |
|483d602a-7062-41f1-b3a7-3eebc9baca88|780.96                  |
|483d602a-7062-41f1-b3a7-3eebc9baca88|M25.541                 |
|483d602a-7062-41f1-b3a7-3eebc9baca88|M85.80                  |
|483d602a-7062-41f1-b3a7-3eebc9baca88|K58.1                   |
|483d602a-7062-41f1-b3a7-3eebc9baca88|R42                     |
|483d602a-7062-41f1-b3a7-3eebc9baca88|724.4                   |
|483d602a-7062-41f1-b3a7-3eebc9baca88|Z96.641                 |
|483d602a-7062-41f1-b3a7-3eebc9baca88|401.9                   |
|483d602a-7062-41f1-b3a7-3eebc9baca88|V04.81                  |
|483d602a-7062-41f1-b3a7-3eebc9baca88|V13.51                  |
|483d602a-7062-41f1-b3a7-3eebc9baca88|M54.41                  |
|483d602a-7062-41f1-b3a7-3eebc9baca88|M1

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [37]:
filtered_df1.show(171,truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+-------------+
|personid                            |comorbidityid|
+------------------------------------+-------------+
|483d602a-7062-41f1-b3a7-3eebc9baca88|M18.11       |
|483d602a-7062-41f1-b3a7-3eebc9baca88|M18.12       |
|483d602a-7062-41f1-b3a7-3eebc9baca88|780.96       |
|483d602a-7062-41f1-b3a7-3eebc9baca88|M25.541      |
|483d602a-7062-41f1-b3a7-3eebc9baca88|M85.80       |
|483d602a-7062-41f1-b3a7-3eebc9baca88|K58.1        |
|483d602a-7062-41f1-b3a7-3eebc9baca88|R42          |
|483d602a-7062-41f1-b3a7-3eebc9baca88|724.4        |
|483d602a-7062-41f1-b3a7-3eebc9baca88|Z96.641      |
|483d602a-7062-41f1-b3a7-3eebc9baca88|401.9        |
|483d602a-7062-41f1-b3a7-3eebc9baca88|V04.81       |
|483d602a-7062-41f1-b3a7-3eebc9baca88|V13.51       |
|483d602a-7062-41f1-b3a7-3eebc9baca88|M54.41       |
|483d602a-7062-41f1-b3a7-3eebc9baca88|M18.11       |
|483d602a-7062-41f1-b3a7-3eebc9baca88|R10.13       |
|483d602a-7062-41f1-b3a7-3eebc9baca88|H43.393 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [22]:
from pyspark.sql.functions import col

# Assuming Como_Result_Final2 is your Spark DataFrame
filtered_df = distinct_df.filter(col("Latest_modified_comorbidityid").rlike(",+"))
filtered_df_cnt1 = filtered_df.count()
print("Count of cohort =",filtered_df_cnt1)
# Show the resulting dataframe
filtered_df.show(truncate=False)

▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+------------------------------------+--------------------+----------------+--------------+-----------------------------+
|personid                            |comorbidityid       |comorbidityid_RF|modified_Commo|Latest_modified_comorbidityid|
+------------------------------------+--------------------+----------------+--------------+-----------------------------+
|ca3663a3-ad63-4b99-b2f2-c4b25281895d|M54.9, G89.29       |G8929           |G89           |G89.9, G89.29                |
|e79911f8-e248-4211-8191-e186ce208ed2|M54.5, G89.29       |G8929           |G89           |G89.5, G89.29                |
|1d556661-f12b-4e52-82a7-649f872f3546|M54.41, G89.29      |G8929           |G89           |G89.41, G89.29               |
|8e484c43-4593-4381-bd22-264a0199e170|S02.119A, S06.9X9A  |S069X9A         |S06           |S06.119A, S06.9X9A           |
|8370cad7-6bb2-473b-ba3e-9b4d9e6459d5|M79.674, M79.89     |M79674          |M79           |M79.674, M79.89              |
|9a3f73c5-617b-4bac-a865

<IPython.core.display.Javascript object>

In [23]:
from pyspark.sql.functions import col

# Assuming final_df is your Spark DataFrame
filtered_df = Como_Result_Final2.filter((col("Como_comorbidityid").rlike(",+")) & (~col("Como_comorbidityid").rlike("^[a-zA-Z]")))
filtered_df_cnt1 = filtered_df.count()
print("Count of cohort =", filtered_df_cnt1)
# Show the resulting dataframe
filtered_df.show(truncate=False)


▸,:,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Count of cohort = 0


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

+-------------+------------------+
|Como_personid|Como_comorbidityid|
+-------------+------------------+
+-------------+------------------+



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Displaying the count of records
record_count1 = Como_Result_Final2.count()
record_count = final_df.count()
record_count2 = combined_addfloatpoint_Final.count()
print("Count of Records:", record_count)
print("Count of Records:", record_count1)
print("Count of Records:", record_count2)

In [ ]:
from pyspark.sql.functions import col

# Assuming Como_Result_Final2 is your Spark DataFrame
filtered_df = Como_Result_Final2.filter(col("personid") == "0053f0b8-4a78-4826-b193-9a92cd4e43d5")

# Display the filtered DataFrame
filtered_df_cnt = filtered_df.count()
print("Count =",filtered_df_cnt)

In [ ]:
filtered_df.show(55, truncate=False)

In [ ]:
from pyspark.sql.functions import col

# Assuming Como_Result_Final2 is your Spark DataFrame
filtered_df1 = combined_addfloatpoint_Final.filter(col("personid") == "0053f0b8-4a78-4826-b193-9a92cd4e43d5")

# Display the filtered DataFrame
filtered_df_cnt1 = filtered_df1.count()
print("Count =",filtered_df_cnt1)

In [ ]:
filtered_df1.show(42, truncate=False)

In [ ]:
from pyspark.sql.functions import col

# Assuming Como_Result_Final2 is your Spark DataFrame
filtered_df2 = final_df.filter(col("personid") == "04fbb148-a952-4b9e-9ba8-4e1a51dee61e")

# Display the filtered DataFrame
filtered_df_cnt2 = filtered_df2.count()
print("Count =",filtered_df_cnt2)

In [ ]:
filtered_df2.show(110, truncate=False)

In [ ]:
combined_addfloatpoint_Final.show(20, truncate=False)